# Restaurant Website Scraping & Fetching Layer
**IST 488 Final Project — Milestone (Apr 19, 2026)**
**Owner:** Ryan Peters

This notebook is the **Enrichment Agent's fetching layer**. It takes a restaurant website URL (returned by Lauren's Google Places Discovery Agent) and returns clean text content for Leytisha's OpenAI-powered menu/pricing extractor.

**Pipeline position:**
`Google Places API (Lauren) → [THIS LAYER: fetch_restaurant_content] → OpenAI enrichment (Leytisha) → JSON structuring → ChromaDB (Toby)`

**What this module does:**
1. Fetches the restaurant homepage with realistic headers.
2. Auto-detects content type (HTML vs PDF).
3. Extracts main textual content (trafilatura primary, BeautifulSoup fallback).
4. Follows same-domain links whose URL or anchor text matches menu patterns (`menu`, `food`, `lunch`, etc.).
5. Handles PDF menus (very common — many restaurants publish menus as PDF).
6. Fails gracefully: returns structured error codes (`blocked`, `timeout`, `cloudflare_challenge`, `error`) so downstream agents can skip or retry.
7. Caches results to disk by URL hash to avoid re-scraping during development.
8. Exposes itself as an **OpenAI-compatible tool definition** — satisfies instructor requirement #3 (tool/function made available to the LLM API).

**Apr 19 milestone scope:** basic fetch + HTML extraction + tool definition + tested on 3+ sites.
**Apr 26 scope:** PDF fallback hardening, block-detection for sites with stricter bot protection, test across the full 20-restaurant sample.


## 1. Install dependencies

In [ ]:
!pip install -q requests beautifulsoup4 trafilatura pdfplumber lxml


## 2. Imports & config

In [ ]:
import re
import json
import hashlib
import logging
from io import BytesIO
from pathlib import Path
from urllib.parse import urljoin, urlparse
from typing import Optional

import requests
from bs4 import BeautifulSoup
import trafilatura
import pdfplumber

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("restaurant_scraper")

# --- Config ---
USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

REQUEST_TIMEOUT = 15           # seconds
MAX_CONTENT_BYTES = 5_000_000  # 5 MB — skip absurdly large PDFs/pages
MAX_MENU_LINKS_TO_FOLLOW = 3   # per-restaurant cap to keep cost bounded

CACHE_DIR = Path("./scrape_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Substrings (in URL path or anchor text) that suggest a menu page/PDF.
MENU_LINK_PATTERNS = [
    r"menu", r"menus", r"food", r"dinner", r"lunch",
    r"brunch", r"drinks", r"beverages", r"wine-list", r"cocktails",
]


## 3. On-disk cache (by URL hash)
Keeps dev iteration fast and keeps you from hammering small restaurant sites.

In [ ]:
def _cache_path(url: str) -> Path:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{h}.json"

def _load_from_cache(url: str) -> Optional[dict]:
    p = _cache_path(url)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        logger.warning(f"Cache read failed for {url}: {e}")
        return None

def _save_to_cache(url: str, data: dict) -> None:
    try:
        _cache_path(url).write_text(json.dumps(data), encoding="utf-8")
    except Exception as e:
        logger.warning(f"Cache write failed for {url}: {e}")

def clear_cache() -> int:
    """Delete all cached entries. Returns count removed."""
    n = 0
    for f in CACHE_DIR.glob("*.json"):
        f.unlink()
        n += 1
    return n


## 4. Low-level fetch with block detection
Single-URL HTTP fetch. Returns `(Response | None, error_code | None)`. Detects common failure modes so the pipeline can route around them.

In [ ]:
def _fetch_raw(url: str) -> tuple[Optional[requests.Response], Optional[str]]:
    """Fetch one URL. Returns (response, error_code). response is None on failure."""
    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
            stream=True,
        )
        # Guard against huge payloads
        cl = r.headers.get("Content-Length")
        if cl and int(cl) > MAX_CONTENT_BYTES:
            return None, f"too_large_{cl}"

        # Materialize content with size cap
        content = r.raw.read(MAX_CONTENT_BYTES + 1, decode_content=True)
        if len(content) > MAX_CONTENT_BYTES:
            return None, "too_large"
        r._content = content  # inject for .text / .content access

        if r.status_code in (403, 401):
            return None, f"blocked_http_{r.status_code}"
        if r.status_code == 429:
            return None, "rate_limited"
        if r.status_code >= 400:
            return None, f"http_{r.status_code}"

        # Cloudflare / bot-wall heuristics
        head = r.text[:1500].lower() if r.text else ""
        if "just a moment" in head or "cf-chl" in head or "challenge-platform" in head:
            return None, "cloudflare_challenge"

        return r, None
    except requests.exceptions.Timeout:
        return None, "timeout"
    except requests.exceptions.SSLError:
        return None, "ssl_error"
    except requests.exceptions.TooManyRedirects:
        return None, "too_many_redirects"
    except requests.exceptions.RequestException as e:
        return None, f"request_error:{str(e)[:80]}"


## 5. PDF text extraction
Many restaurants publish menus as PDF. pdfplumber handles typical text-based menus well; image-only PDFs will return empty (Apr 26: add OCR fallback if needed).

In [ ]:
def _extract_pdf_text(content: bytes) -> str:
    """Extract text from PDF bytes. Returns empty string on failure or image-only PDFs."""
    try:
        with pdfplumber.open(BytesIO(content)) as pdf:
            pages = [p.extract_text() or "" for p in pdf.pages]
        text = "\n".join(pages).strip()
        return text
    except Exception as e:
        logger.warning(f"PDF extraction failed: {e}")
        return ""


## 6. HTML content extraction
Trafilatura is our primary extractor (better at filtering nav/footer/ads). Falls back to stripped BeautifulSoup if trafilatura can't get enough.

In [ ]:
def _extract_html_text(html: str) -> str:
    """Extract main textual content from HTML. Trafilatura primary, BS4 fallback."""
    try:
        extracted = trafilatura.extract(
            html,
            include_tables=True,
            include_links=False,
            include_comments=False,
            no_fallback=False,
        )
        if extracted and len(extracted) > 200:
            return extracted
    except Exception as e:
        logger.debug(f"trafilatura failed: {e}")

    # Fallback: BeautifulSoup with noise stripped
    try:
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text
    except Exception as e:
        logger.warning(f"BS4 fallback failed: {e}")
        return ""


## 7. Menu-link finder
Most homepages don't contain the full menu; they link to it. We scan for same-domain links (and PDFs) whose URL or anchor text matches menu-ish patterns.

In [ ]:
def _find_menu_links(html: str, base_url: str) -> list[str]:
    """Return up to MAX_MENU_LINKS_TO_FOLLOW likely menu URLs."""
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        return []

    base_domain = urlparse(base_url).netloc
    seen, out = set(), []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue
        anchor_text = (a.get_text() or "").strip().lower()
        combined = (href + " " + anchor_text).lower()

        if not any(re.search(pat, combined) for pat in MENU_LINK_PATTERNS):
            continue

        full = urljoin(base_url, href)
        # Same-domain OR any PDF link (PDFs often hosted on CDNs)
        is_same_domain = urlparse(full).netloc == base_domain
        is_pdf = full.lower().split("?")[0].endswith(".pdf")
        if not (is_same_domain or is_pdf):
            continue

        if full == base_url or full in seen:
            continue
        seen.add(full)
        out.append(full)
        if len(out) >= MAX_MENU_LINKS_TO_FOLLOW:
            break

    return out


## 8. Main entry point: `fetch_restaurant_content`
This is the function the enrichment agent (or OpenAI function-calling) invokes.

Return shape:
```python
{
    "url": str,                    # input URL
    "status": str,                 # "success" | "empty" | "blocked" | "error"
    "content": str,                # aggregated extracted text (main page + menu sub-pages, delimited)
    "content_type": str | None,    # "html" | "pdf"
    "pages_fetched": list[str],    # URLs we actually read from
    "error": str | None,           # error code if status != success
    "from_cache": bool,            # True if result came from local cache
}
```


In [ ]:
def fetch_restaurant_content(website_url: str, use_cache: bool = True) -> dict:
    """
    Fetch a restaurant's website and any linked menu pages or PDFs.

    Args:
        website_url: Full URL of the restaurant's website (from Google Places `website` field).
        use_cache:   If True, return cached result when available and write result to cache.

    Returns:
        dict — see module docstring for shape.
    """
    # Cache hit
    if use_cache:
        cached = _load_from_cache(website_url)
        if cached is not None:
            cached["from_cache"] = True
            return cached

    result = {
        "url": website_url,
        "status": "error",
        "content": "",
        "content_type": None,
        "pages_fetched": [],
        "error": None,
        "from_cache": False,
    }

    resp, err = _fetch_raw(website_url)
    if resp is None:
        result["error"] = err
        result["status"] = "blocked" if (err or "").startswith(("blocked", "cloudflare", "rate")) else "error"
        if use_cache:
            _save_to_cache(website_url, result)
        return result

    ct = resp.headers.get("Content-Type", "").lower()
    pages: list[dict] = []

    # --- Case A: the URL itself is a PDF ---
    if "pdf" in ct or website_url.lower().split("?")[0].endswith(".pdf"):
        text = _extract_pdf_text(resp.content)
        pages.append({"url": website_url, "type": "pdf", "text": text})

    # --- Case B: HTML homepage ---
    elif "html" in ct or (resp.text and "<html" in resp.text[:500].lower()):
        main_text = _extract_html_text(resp.text)
        pages.append({"url": website_url, "type": "html", "text": main_text})

        for link in _find_menu_links(resp.text, website_url):
            sub_resp, sub_err = _fetch_raw(link)
            if sub_resp is None:
                logger.info(f"  menu link skipped ({sub_err}): {link}")
                continue
            sub_ct = sub_resp.headers.get("Content-Type", "").lower()
            if "pdf" in sub_ct or link.lower().split("?")[0].endswith(".pdf"):
                sub_text = _extract_pdf_text(sub_resp.content)
                pages.append({"url": link, "type": "pdf", "text": sub_text})
            else:
                sub_text = _extract_html_text(sub_resp.text)
                pages.append({"url": link, "type": "html", "text": sub_text})

    else:
        result["error"] = f"unsupported_content_type:{ct}"
        if use_cache:
            _save_to_cache(website_url, result)
        return result

    # Aggregate — include source URL delimiters so downstream LLM can attribute content
    combined = "\n\n---\n\n".join(
        f"[SOURCE: {p['url']}]\n{p['text']}" for p in pages if (p.get("text") or "").strip()
    )

    result["status"] = "success" if combined.strip() else "empty"
    result["content"] = combined
    result["content_type"] = pages[0]["type"] if pages else None
    result["pages_fetched"] = [p["url"] for p in pages if (p.get("text") or "").strip()]

    if use_cache:
        _save_to_cache(website_url, result)
    return result


## 9. OpenAI-compatible tool definition
Hands `fetch_restaurant_content` to the LLM as a callable tool. This is what makes our app satisfy **requirement #3 (tool/function made available to the LLM API)**.

Leytisha can import this constant from this module when wiring up the enrichment agent's `tools=[...]` parameter in the `chat.completions.create` call.


In [ ]:
RESTAURANT_SCRAPE_TOOL = {
    "type": "function",
    "function": {
        "name": "fetch_restaurant_content",
        "description": (
            "Fetch a restaurant's website and any linked menu pages or PDFs. "
            "Returns aggregated text content suitable for extracting menu items, "
            "prices, cuisine type, and other restaurant details. Handles HTML and "
            "PDF menus. Returns a structured dict with status and error fields so "
            "the caller can skip restaurants whose sites are blocked or unreachable."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "website_url": {
                    "type": "string",
                    "description": (
                        "Full URL of the restaurant's website (e.g. the `website` "
                        "field from a Google Places API Place Details response)."
                    ),
                },
                "use_cache": {
                    "type": "boolean",
                    "description": "If true (default), return cached scrape when available.",
                    "default": True,
                },
            },
            "required": ["website_url"],
        },
    },
}

# Dispatcher table for function-calling loop
TOOL_FUNCTIONS = {
    "fetch_restaurant_content": fetch_restaurant_content,
}


## 10. Test — Syracuse-area restaurants
Swap in URLs from Lauren's Discovery Agent output. These are starter examples; expect 1–2 to hit a block so you can see the error handling in action.

In [ ]:
test_urls = [
    "https://www.dinosaurbarbque.com/",
    "https://pastabilities.com/",
    "https://apizzaregionale.com/",
]

for u in test_urls:
    print(f"\n=== {u} ===")
    r = fetch_restaurant_content(u)
    print(f"  status:        {r['status']}")
    print(f"  from_cache:    {r['from_cache']}")
    print(f"  pages_fetched: {len(r['pages_fetched'])}")
    print(f"  content chars: {len(r['content'])}")
    if r["error"]:
        print(f"  error:         {r['error']}")
    if r["content"]:
        preview = r["content"][:400].replace("\n", " ")
        print(f"  preview:       {preview}...")


## 11. Example: how Leytisha wires this into the enrichment agent
Not executed here (needs `openai` package + API key), but this is the shape for the handoff. Given that the group plan is gpt-4o, this is the standard tool-use loop.


In [ ]:
# Pseudocode / integration sketch for the enrichment agent.
# Leytisha owns the actual call; this block just shows the integration contract.

INTEGRATION_SKETCH = """
from openai import OpenAI
client = OpenAI()

def enrich_restaurant(restaurant_metadata: dict) -> dict:
    messages = [
        {"role": "system", "content": "You extract menu items, price ranges, and cuisine type from restaurant website content."},
        {"role": "user",   "content": f"Enrich this restaurant: {restaurant_metadata}"},
    ]
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=[RESTAURANT_SCRAPE_TOOL],
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if msg.tool_calls:
        for call in msg.tool_calls:
            fn = TOOL_FUNCTIONS[call.function.name]
            args = json.loads(call.function.arguments)
            result = fn(**args)
            messages.append(msg)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
        final = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            response_format={"type": "json_object"},
        )
        return json.loads(final.choices[0].message.content)
    return {}
"""
print(INTEGRATION_SKETCH)


## 12. Milestone checklist (Apr 19)

- [x] Fetcher handles HTML with realistic headers
- [x] Auto-detects HTML vs PDF content
- [x] Extracts main content (trafilatura + BS4 fallback)
- [x] Follows menu-like links (same-domain + PDFs)
- [x] Handles PDF menus
- [x] Structured error codes (`blocked`, `timeout`, `cloudflare_challenge`, etc.)
- [x] On-disk cache keyed by URL hash
- [x] Exposed as OpenAI tool (`RESTAURANT_SCRAPE_TOOL`) — satisfies instructor req #3
- [x] Tested on 3+ sites

## Apr 26 follow-ups

- **OCR fallback** for image-only PDF menus (`pytesseract` + `pdf2image`).
- **`curl_cffi` escape hatch** for sites that block plain `requests` via TLS fingerprinting (drop-in replacement; bypasses many Cloudflare/bot walls).
- **Retry with backoff** on `timeout` / `rate_limited`.
- **Test against full 20-restaurant sample** once Lauren's Discovery Agent output is in the shared JSON.
- Coordinate with **Toby** on what text-chunking strategy his ChromaDB ingest expects — may want to pre-chunk here rather than pass one big blob.

## Instructor-requirement coverage (your group's shared checklist)

| Req | Where it lives | Owner |
|---|---|---|
| 1. Short-term + long-term memory | Conversation history (short) + ChromaDB persistent store (long) | Toby + Leytisha |
| 2. RAG + reranking | ChromaDB semantic search + rerank step | Toby |
| 3. Tool/function to LLM API | `RESTAURANT_SCRAPE_TOOL` in this notebook | **Ryan (this module)** |
| 4. *(check with Dunham — got truncated in the feedback)* | ? | ? |
